# MNIST Classification with Fully Connected Layers in PyTorch

This notebook investigates how the **composition of input features** affects model performance through three experiments:

| Case | Input | Key idea |
|------|-------|----------|
| **1 – Zero padding** | 784 real + 784 zeros (1 568-dim) | Baseline sanity-check: dead dimensions shouldn't hurt |
| **2 – Noise padding** | 784 real + 784 white noise (1 568-dim) | Extra random features → overfitting |
| **3 – Ridge-regularised noise** | Same as Case 2 + L2 weight penalty | Regularisation combats overfitting |

Each case trains the *same* 3-layer FC architecture for 15 epochs and ends with a side-by-side comparison.


## 1. Setup & Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Base MNIST Data

We first download and cache the raw 784-dim MNIST tensors so every case augments
from the same pre-processed features.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # scale pixels to [-1, 1]
])

mnist_train = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
mnist_test  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Pre-load everything into flat tensors for easy augmentation later
def to_flat_tensors(dataset):
    loader = DataLoader(dataset, batch_size=1024, shuffle=False)
    X_list, y_list = [], []
    for imgs, labels in loader:
        X_list.append(imgs.view(imgs.size(0), -1))   # (B, 784)
        y_list.append(labels)
    return torch.cat(X_list), torch.cat(y_list)

X_train_base, y_train = to_flat_tensors(mnist_train)
X_test_base,  y_test  = to_flat_tensors(mnist_test)

print(f"Training features : {X_train_base.shape}")
print(f"Test features     : {X_test_base.shape}")


In [ ]:
# Quick visual sanity check
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_base[i].view(28, 28), cmap="gray")
    ax.set_title(f"Label: {y_train[i].item()}")
    ax.axis("off")
plt.suptitle("Sample MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()


## 3. Shared Model & Training Utilities

All three cases reuse the same FC architecture and training loop.  
The only differences are the **input dimension** and the **weight-decay** hyperparameter.

In [ ]:
class MNISTClassifier(nn.Module):
    """3-layer FC network.  input_size is set per experiment."""
    def __init__(self, input_size=784, hidden1=512, hidden2=256,
                 num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


def make_loaders(X_train, X_test, y_train, y_test, batch_size=64):
    train_ds = TensorDataset(X_train, y_train)
    test_ds  = TensorDataset(X_test,  y_test)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(test_ds,  batch_size=256,        shuffle=False))


def l1_penalty(model):
    return sum(p.abs().sum() for p in model.parameters())


def train_one_epoch(model, loader, optimizer, criterion, l1_lambda=0.0):
    model.train()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y) + l1_lambda*l1_penalty(model)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            total_loss += criterion(out, y).item() * X.size(0)
            correct    += (out.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def run_experiment(
    X_train, X_test, y_train, y_test,
    weight_decay=0.0, l1_lambda=0.0,
    num_epochs=15, label="Experiment"
):
    """Train a model and return its history dict."""
    input_size = X_train.shape[1]
    model      = MNISTClassifier(input_size=input_size).to(device)
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    # scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    train_loader, test_loader = make_loaders(X_train, X_test, y_train, y_test)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    print(f"\n{'='*60}")
    print(f"  {label}  (input_dim={input_size}, weight_decay={weight_decay})")
    print(f"{'='*60}")
    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc = evaluate(model, test_loader, criterion)
        # scheduler.step()
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(f"  Epoch {epoch:2d}/{num_epochs} | "
              f"Train Loss {tr_loss:.4f}  Acc {tr_acc*100:.2f}% | "
              f"Val Loss {va_loss:.4f}  Acc {va_acc*100:.2f}%")
    print(f"  ➜ Final val accuracy: {history['val_acc'][-1]*100:.2f}%")
    return history, model


---
## Case 1 – Zero-Padded Inputs

We append **784 constant-zero dimensions** to every sample, doubling the input
size from 784 to 1 568.

**What to expect:** The model must learn to ignore the dead columns. Because the
extra weights always receive a gradient of zero they stay near their random
initialisation and the network should still converge to roughly the same accuracy
as a plain 784-dim model. This is the "harmless baseline" for the noise experiment.

In [ ]:
# Append 784 zeros to every sample
zeros_train = torch.zeros(X_train_base.size(0), 784*3)
zeros_test  = torch.zeros(X_test_base.size(0),  784*3)

X_train_zero = torch.cat([X_train_base, zeros_train], dim=1)   # (60000, 1568)
X_test_zero  = torch.cat([X_test_base,  zeros_test],  dim=1)   # (10000, 1568)

print(f"Augmented training shape : {X_train_zero.shape}")
print(f"Augmented test shape     : {X_test_zero.shape}")
print(f"Extra-dimension stats    : mean={zeros_train.mean():.4f}, std={zeros_train.std():.4f}")


In [ ]:
history_zero, model_zero = run_experiment(
    X_train_zero, X_test_zero, y_train, y_test,
    weight_decay=0.0,
    label="Case 1 – Zero Padding"
)


In [ ]:
NUM_EPOCHS = len(history_zero["train_acc"])
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, history_zero["train_loss"], label="Train")
ax1.plot(epochs, history_zero["val_loss"],   label="Validation")
ax1.set_title("Case 1 – Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, [a*100 for a in history_zero["train_acc"]], label="Train")
ax2.plot(epochs, [a*100 for a in history_zero["val_acc"]],   label="Validation")
ax2.set_title("Case 1 – Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.legend()
plt.suptitle("Case 1: Zero-Padded Inputs", fontsize=13)
plt.tight_layout(); plt.show()


---
## Case 2 – White-Noise Padded Inputs

We append **784 dimensions of i.i.d. Gaussian noise** (mean 0, std 1).  
The noise is sampled once and fixed, so the model can in principle memorise it.

**What to expect:** The network now has 784 extra "features" that correlate with
nothing real. A sufficiently large model can still overfit — it memorises
which noise pattern happens to co-occur with which class label in the *training*
set. You should see the training accuracy pulling ahead of validation accuracy
more than in Case 1.

In [ ]:
torch.manual_seed(42)

# Fixed noise matrices (same noise applied to every epoch)
noise_train = torch.randn(X_train_base.size(0), 784*3)
noise_test  = torch.randn(X_test_base.size(0),  784*3)

X_train_noise = torch.cat([X_train_base, noise_train], dim=1)
X_test_noise  = torch.cat([X_test_base,  noise_test],  dim=1)

print(f"Augmented training shape : {X_train_noise.shape}")
print(f"Extra-dimension stats    : mean={noise_train.mean():.4f}, std={noise_train.std():.4f}")


In [ ]:
history_noise, model_noise = run_experiment(
    X_train_noise, X_test_noise, y_train, y_test,
    weight_decay=0.0,
    label="Case 2 – White-Noise Padding"
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, history_noise["train_loss"], label="Train")
ax1.plot(epochs, history_noise["val_loss"],   label="Validation")
ax1.set_title("Case 2 – Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, [a*100 for a in history_noise["train_acc"]], label="Train")
ax2.plot(epochs, [a*100 for a in history_noise["val_acc"]],   label="Validation")
ax2.set_title("Case 2 – Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.legend()
plt.suptitle("Case 2: White-Noise Padded Inputs", fontsize=13)
plt.tight_layout(); plt.show()


---
## Case 3 – Ridge Regularisation on Noisy Inputs

We reuse the **same noisy data** from Case 2 but add an **L2 weight penalty**
(also called *ridge regression* or *weight decay*) to the Adam optimiser.

### Why L2 / Ridge?

The L2 penalty adds a term $\frac{\lambda}{2}\|\mathbf{w}\|^2$ to the loss,
which penalises large weights uniformly.  Weights connected to the noisy
dimensions cannot grow large enough to "memorise" training-set noise patterns,
so the gap between training and validation accuracy should narrow.

We pick `weight_decay = 1e-3` — large enough to matter but small enough not to
destroy the signal features.

In [ ]:
RIDGE_LAMBDA = 1e-3

history_ridge, model_ridge = run_experiment(
    X_train_noise, X_test_noise, y_train, y_test,
    weight_decay=RIDGE_LAMBDA,
    label=f"Case 3 – Ridge (λ={RIDGE_LAMBDA})"
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, history_ridge["train_loss"], label="Train")
ax1.plot(epochs, history_ridge["val_loss"],   label="Validation")
ax1.set_title("Case 3 – Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, [a*100 for a in history_ridge["train_acc"]], label="Train")
ax2.plot(epochs, [a*100 for a in history_ridge["val_acc"]],   label="Validation")
ax2.set_title("Case 3 – Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.legend()
plt.suptitle("Case 3: Ridge-Regularised Noisy Inputs", fontsize=13)
plt.tight_layout(); plt.show()


## Lasso

In [ ]:
history_lasso, model_lasso = run_experiment(
    X_train_noise, X_test_noise, y_train, y_test,
    weight_decay=0.0, l1_lambda=1e-3,
    label="Case 4 – Lasso Padding"
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, history_lasso["train_loss"], label="Train")
ax1.plot(epochs, history_lasso["val_loss"], label="Validation")
ax1.set_title("Case 4 – Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, [a*100 for a in history_lasso["train_acc"]], label="Train")
ax2.plot(epochs, [a*100 for a in history_lasso["val_acc"]],   label="Validation")
ax2.set_title("Case 4 – Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.legend()
plt.suptitle("Case 4: Lasso term", fontsize=13)
plt.tight_layout(); plt.show()


---
## 4. Side-by-Side Comparison

Let's overlay all three cases to make the key dynamics visible:

- **Train vs Val gap** reveals overfitting.
- **Val accuracy** shows which approach actually generalises.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {"zero": "#1f77b4", "noise": "#d62728", "ridge": "#2ca02c"}

# --- Validation accuracy ---
ax = axes[0]
ax.plot(epochs, [a*100 for a in history_zero["val_acc"]],
        color=colors["zero"],  label="Case 1 – Zero Pad (val)")
ax.plot(epochs, [a*100 for a in history_noise["val_acc"]],
        color=colors["noise"], label="Case 2 – Noise (val)")
ax.plot(epochs, [a*100 for a in history_ridge["val_acc"]],
        color=colors["ridge"], label="Case 3 – Ridge (val)")
ax.set_title("Validation Accuracy"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.legend(); ax.grid(True, alpha=0.3)

# --- Train-Val gap (overfitting signal) ---
ax = axes[1]
gap_zero  = [100*(tr - va) for tr, va in
             zip(history_zero["train_acc"],  history_zero["val_acc"])]
gap_noise = [100*(tr - va) for tr, va in
             zip(history_noise["train_acc"], history_noise["val_acc"])]
gap_ridge = [100*(tr - va) for tr, va in
             zip(history_ridge["train_acc"], history_ridge["val_acc"])]

ax.plot(epochs, gap_zero,  color=colors["zero"],  label="Case 1 – Zero Pad")
ax.plot(epochs, gap_noise, color=colors["noise"], label="Case 2 – Noise")
ax.plot(epochs, gap_ridge, color=colors["ridge"], label="Case 3 – Ridge")
ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_title("Train – Val Accuracy Gap (overfitting signal)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Gap (pp)")
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("Three-Way Comparison", fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# Numeric summary
print(f"{'Case':<40} {'Final Train Acc':>16} {'Final Val Acc':>14} {'Gap':>8}")
print("-" * 82)
for label, h in [("Case 1 – Zero Padding",          history_zero),
                 ("Case 2 – White Noise (no reg.)",  history_noise),
                 ("Case 3 – Ridge (λ=1e-3)",         history_ridge)]:
    tr  = h["train_acc"][-1] * 100
    va  = h["val_acc"][-1]   * 100
    gap = tr - va
    print(f"{label:<40} {tr:>14.2f}%  {va:>12.2f}%  {gap:>6.2f}pp")


---
## 5. Key Takeaways

| Observation | Explanation |
|-------------|-------------|
| **Case 1 ≈ Case 3 (val accuracy)** | Zero dimensions are truly inert — they cannot be memorised. |
| **Case 2 has the largest train–val gap** | Fixed random noise provides spurious correlations the model can memorise. |
| **Case 3 closes that gap** | Ridge regularisation penalises large weights, preventing the model from over-relying on noisy inputs. |
| **Ridge is easy to apply in PyTorch** | Simply pass `weight_decay=λ` to the optimizer — no custom loss required. |

### When to prefer Ridge vs Lasso

- **Ridge (L2)** shrinks all weights smoothly towards zero — weights on noisy 
  features become small but non-zero. Good default choice.
- **Lasso (L1)** drives some weights to exactly zero (sparse solution) — in 
  principle it could *eliminate* the noise dimensions entirely. However, L1 is 
  not natively supported by PyTorch optimisers and requires a custom penalty term.
